# 36 — Long-Term Memory Agent

Persist user facts across conversation threads using LangGraph's `InMemoryStore`. Each session retrieves relevant memories, responds with that context injected, then extracts and stores new facts — so the agent gets smarter across threads.

**What you'll learn**
- `InMemoryStore` — cross-thread key-value store (persists beyond a single `invoke`)
- `store.put(namespace, key, value)` — write a fact to memory
- `store.search(namespace, query, limit)` — semantic search over stored values
- Namespace pattern: `("memories", user_id)` isolates facts per user
- Retrieve → respond → store loop as a LangGraph graph

**Contrast:** Standard LangGraph state resets on every `invoke`. `InMemoryStore` breaks that boundary — data written in thread-1 is visible in thread-2.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langgraph python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Core concept: InMemoryStore ───────────────────────────────────────────
#
# Standard LangGraph state:              InMemoryStore:
#   thread-1 → invoke → state gone         thread-1 → store.put → persists
#   thread-2 → invoke → fresh state        thread-2 → store.search → retrieves
#
# store.put(namespace, key, value)
#   namespace: tuple like ("memories", "user-alice") scopes data per user
#   key:       unique string within namespace
#   value:     any dict -- we use {"fact": "User is a Python developer"}
#
# store.search(namespace, query=str, limit=N)
#   returns list of SearchResult with .value field
#   semantic search: finds relevant facts even if keywords differ

from langgraph.store.memory import InMemoryStore

demo_store = InMemoryStore()
ns = ("memories", "demo-user")

demo_store.put(ns, "fact-0", {"fact": "User is a Python developer"})
demo_store.put(ns, "fact-1", {"fact": "User prefers concise responses"})
demo_store.put(ns, "fact-2", {"fact": "User is building a LangGraph agent"})

results = demo_store.search(ns, query="what does the user build?", limit=3)
print("Retrieved facts:")
for r in results:
    print(f"  - {r.value['fact']}")

In [ ]:
# ── 4. Build the long-term memory graph ──────────────────────────────────────
import json
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.store.memory import InMemoryStore

USER_ID = "user-alice"
NAMESPACE = ("memories", USER_ID)
store = InMemoryStore()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

SYSTEM_PROMPT = (
    "You are a helpful assistant with long-term memory about the user.\n"
    "Retrieved memories:\n{memories}\n\n"
    "Use these to personalize your response."
)


class MemoryState(TypedDict):
    thread_id: str
    messages: list[str]
    memories: list[str]
    response: str


def retrieve_memories(state: MemoryState) -> dict:
    query = state["messages"][-1] if state["messages"] else ""
    results = store.search(NAMESPACE, query=query, limit=5)
    return {"memories": [r.value["fact"] for r in results]}


def respond(state: MemoryState) -> dict:
    mem_text = "\n".join(f"- {m}" for m in state["memories"]) or "None yet"
    system = SYSTEM_PROMPT.format(memories=mem_text)
    history = [HumanMessage(content=msg) for msg in state["messages"]]
    resp = llm.invoke([SystemMessage(content=system)] + history)
    return {"response": resp.content}


def store_memories(state: MemoryState) -> dict:
    prompt = (
        f"Extract 1-3 facts about the user from this conversation.\n"
        f"Messages: {state['messages']}\nResponse: {state['response']}\n"
        f"Return only a JSON array of strings."
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    try:
        facts = json.loads(result.content)
        for i, fact in enumerate(facts[:3]):
            store.put(NAMESPACE, f"{state['thread_id']}-fact-{i}", {"fact": fact})
            print(f"  Stored: {fact}")
    except (json.JSONDecodeError, TypeError):
        pass
    return {}


graph = StateGraph(MemoryState)
graph.add_node("retrieve", retrieve_memories)
graph.add_node("respond", respond)
graph.add_node("store", store_memories)
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "respond")
graph.add_edge("respond", "store")
graph.add_edge("store", END)
app = graph.compile()
print("Graph compiled.")

In [ ]:
# ── 5. Run: thread-1 introduces user, thread-2 uses that memory ───────────────
CONVERSATIONS = [
    (
        "thread-1",
        [
            "Hi! I'm Alice. I'm a Python developer working on LangGraph.",
            "I prefer concise responses and love building agentic systems.",
        ],
    ),
    (
        "thread-2",
        [
            "What do you know about me?",
            "Can you recommend a next step for my LangGraph project?",
        ],
    ),
]

for thread_id, messages in CONVERSATIONS:
    print(f"\n{'=' * 60}")
    print(f"Thread: {thread_id}")
    result = app.invoke(
        {
            "thread_id": thread_id,
            "messages": messages,
            "memories": [],
            "response": "",
        }
    )
    print(f"Memories retrieved: {result['memories']}")
    print(f"Response: {result['response'][:300]}")

## Exercises

**Exercise 1 — Inspect the store:** After thread-1, call `store.search(NAMESPACE, query="Python", limit=10)` and print all stored facts. What did the LLM extract?

**Exercise 2 — Multiple users:** Add a second user (`user-bob`) with their own namespace. Verify that Alice's memories don't appear in Bob's retrieve results.

**Exercise 3 — Memory decay:** Add a `timestamp` field to stored facts. In the retrieve step, filter out facts older than N seconds to simulate memory decay.

**Exercise 4 — Persistent store:** Swap `InMemoryStore` for LangGraph's `SqliteStore` (if available) so memories survive process restarts.

## What's next

- **11-hitl-approval** — add human-in-the-loop interrupts before storing memories
- **39-checkpoint-resume** — SqliteSaver for persisting graph state (not just memory)
- **6-multi-agent-graph** — multiple agents sharing the same store